In [8]:
import os, pickle
import numpy as np
import pandas as pd

from models import carl, taylr
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import c6

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [ ]:
torch.set_float32_matmul_precision('high')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [10]:
run_dir = 'run/h4l/'
data_dir = 'data/'

taylr_ckpts = [[30,0.08],[21,0.15],[25,0.08],[194,0.05]]
n_coeffs = len(taylr_ckpts)

taylr_dirs = [f'taylr_{i+1}/' for i in range(n_coeffs)]

logs_dir = 'lightning_logs/version_0'

events_gg_file = 'ggZZ2e2m_sbi/events.csv'
events_qq_file = 'qqZZ2e2m/events.csv'

taylr_ckpt_paths = [os.path.join(run_dir, taylr_dirs[i], logs_dir, 'checkpoints', f'epoch={taylr_ckpts[i][0]}-val_loss={taylr_ckpts[i][1]}.ckpt') for i in range(n_coeffs)]

events_gg_path = os.path.join(data_dir, events_gg_file)
events_qq_path = os.path.join(data_dir, events_qq_file)

In [ ]:
sample_size = 100000

features = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
            'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
            'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
            'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

batch_size = 512

lumi = 3000

c6_linspace = [-20,20,201]
cH_linspace = [-0.02,0.05,141]

c6_space = np.linspace(*c6_linspace)
cH_space = np.linspace(*cH_linspace)

c6_val_asimov = 0.0
cH_val_asimov = 0.0

In [ ]:
xs_gg = 1.5569109*2*1.83

events_gg = mcfm.from_csv(cross_section=xs_gg, file_path=events_gg_path)
events_gg = events_gg.sample(sample_size)
events_gg = zz4l.analyze(events_gg)

w_gg_sm = events_gg.weights.to_numpy()
sigma_gg_sm = np.sum(w_gg_sm)

In [13]:
c6_mod = c6.Modifier(baseline=msq.Component.SBI, events=events_gg, c6_values=[-10, -5, 0, 5, 10])
w_gg_c6, p_gg_c6 = c6_mod.modify(c6=c6_space)

coeffs_true = c6_mod.coefficients[:,1:]

i_c6_asimov = np.where(c6_space==c6_val_asimov)[0][0]
i_c6_sm = np.where(c6_space==0.0)[0][0]

i_cH_asimov = np.where(np.round(cH_space,5)==cH_val_asimov)[0][0]
i_cH_sm = np.where(np.round(cH_space,5)==0.0)[0][0]

In [14]:
trainer = L.Trainer(accelerator='gpu', devices=1)
models_taylr = [taylr.TAYLR.load_from_checkpoint(checkpoint_path=path) for path in taylr_ckpt_paths]

kinematics_gg = events_gg.kinematics[features]

dls_taylr = []
for i in range(n_coeffs):
    with open(os.path.join(run_dir, taylr_dirs[i], 'scaler_X.pkl'), 'rb') as f:
        scaler_X = pickle.load(f)
    X_taylr = scaler_X.transform(kinematics_gg.to_numpy())
    X_taylr = torch.tensor(X_taylr, dtype=torch.float32) 
    y_taylr = torch.tensor(np.ones(kinematics_gg.shape[0]))
    w_taylr = torch.tensor(np.ones(kinematics_gg.shape[0]))
    dl = DataLoader(TensorDataset(X_taylr, y_taylr, w_taylr), batch_size=batch_size, num_workers=8)
    dls_taylr.append(dl)

coeffs_pred = []
for i in range(n_coeffs):
    with open(os.path.join(run_dir, taylr_dirs[i], 'scaler_y.pkl'), 'rb') as f:
        scaler_y = pickle.load(f)
    coeffs_pred.append(scaler_y.inverse_transform(torch.concatenate(trainer.predict(models_taylr[i], dls_taylr[i])).numpy()[:,np.newaxis]).flatten())
coeffs_pred = np.array(coeffs_pred).T

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIB

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [15]:
def f(c6_space, coeffs):
    coefficients = np.concatenate([np.ones((len(coeffs),1)), coeffs], axis=1)
    c6_matrix = np.vander(c6_space, coefficients.shape[1], increasing=True).T
    return np.dot(coefficients, c6_matrix)

In [16]:
# true estimate
w_gg_c6 = w_gg_sm[:,np.newaxis] * f(c6_space, coeffs_true)
w_gg_bsm = w_gg_c6[:,:,np.newaxis] - 2 * (w_gg_sm[:,np.newaxis,np.newaxis] * cH_space[np.newaxis,np.newaxis,:])
sigma_gg_bsm = np.sum(w_gg_bsm, axis=0)

# NSBI estimate
w_gg_c6_nsbi = w_gg_sm[:,np.newaxis] * f(c6_space, coeffs_pred)
w_gg_bsm_nsbi = w_gg_c6_nsbi[:,:,np.newaxis] - 2 * (w_gg_sm[:,np.newaxis,np.newaxis] * cH_space[np.newaxis,np.newaxis,:])
sigma_gg_bsm_nsbi = np.sum(w_gg_bsm_nsbi, axis=0)  # c6 & cH

In [17]:
nu_gg_sm = sigma_gg_sm * lumi
nu_gg_bsm = sigma_gg_bsm * lumi
nu_gg_bsm_nsbi = sigma_gg_bsm_nsbi * lumi

n_gg_asimov = sigma_gg_bsm[i_c6_asimov,i_cH_asimov]*lumi

In [18]:
t_rate = -2 * nu_gg_sm * (np.log(nu_gg_bsm) - np.log(nu_gg_sm)) + 2 * (nu_gg_bsm - nu_gg_sm) 
t_rate_nsbi = -2 * nu_gg_sm * (np.log(nu_gg_bsm_nsbi) - np.log(nu_gg_sm)) + 2 * (nu_gg_bsm_nsbi - nu_gg_sm) 

In [19]:
p_ratio =  (w_gg_bsm / sigma_gg_bsm) / (w_gg_sm[:,np.newaxis,np.newaxis] / sigma_gg_sm)
p_ratio_nsbi = (w_gg_bsm_nsbi / sigma_gg_bsm_nsbi) / (w_gg_sm[:,np.newaxis,np.newaxis] / sigma_gg_sm)

In [20]:
t_shape = - 2 * np.sum(lumi * w_gg_sm[:,np.newaxis,np.newaxis] * np.log(p_ratio) , axis=0)
t_shape_nsbi = - 2 * np.sum(lumi * w_gg_sm[:,np.newaxis,np.newaxis] * np.log(p_ratio_nsbi) , axis=0)

/tmp/ipykernel_920118/4908265.py:1: RuntimeWarning: invalid value encountered in log
  t_shape = - 2 * np.sum(lumi * w_gg_sm[:,np.newaxis,np.newaxis] * np.log(p_ratio) , axis=0)


In [21]:
t = t_rate + t_shape 
t_nsbi = t_rate_nsbi + t_shape_nsbi

In [22]:
print(t_shape_nsbi[i_c6_sm,i_cH_sm])

3.6771686388887387e-10


In [23]:
i_c6_fit = np.nanargmin(t) // cH_linspace[2]
i_cH_fit = np.nanargmin(t)  % cH_linspace[2]

i_c6_fit_nsbi = np.nanargmin(t_nsbi) // cH_linspace[2]
i_cH_fit_nsbi = np.nanargmin(t_nsbi)  % cH_linspace[2]

i_c6_fit_rate = np.nanargmin(t_rate) // cH_linspace[2]
i_cH_fit_rate = np.nanargmin(t_rate)  % cH_linspace[2]

t_min = t[i_c6_fit,i_cH_fit]
t_min_nsbi = t_nsbi[i_c6_fit_nsbi,i_cH_fit_nsbi]
t_min_rate = t_rate[i_c6_fit_rate,i_cH_fit_rate]

In [24]:
print(t_nsbi[i_c6_sm,i_cH_sm])
print(t_nsbi[i_c6_fit_nsbi,i_cH_fit_nsbi])

3.2616482739624993e-10
3.2616482739624993e-10


In [25]:
X, Y = np.meshgrid(c6_space, cH_space)
Z = t.T
Z_nsbi = t_nsbi.T
Z_rate = t_rate.T

In [60]:
fig = plt.figure(figsize=(6,4))

contours_rate = plt.contour(X, Y, Z_rate, levels=[t_min_rate+1,t_min_rate+4], colors='lightgrey', linestyles=['--','-'])
contours_nsbi = plt.contour(X, Y, Z_nsbi, levels=[t_min_nsbi+1,t_min_nsbi+4], colors='black', linestyles=['--','-'])
contours = plt.contour(X, Y, Z, levels=[t_min+1,t_min+4], colors='royalblue', linestyles=['--','-'])

# plt.clabel(contours, fmt=dict(zip([t_min+1,t_min+4], ['$1\sigma$', '$2\sigma$'])))

plt.scatter(c6_space[i_c6_fit], cH_space[i_cH_fit], marker='x', color='royalblue')
# plt.scatter(c6_space[i_c6_fit_nsbi], cH_space[i_cH_fit_nsbi], marker='x', color='black')

labels = [
    Line2D([0],[0],color='royalblue',label='$\\mathrm{True\\ likelihood}$'),
    Line2D([0],[0],color='black',label='$\\mathrm{NSBI\\ estimate}$'),
    Line2D([0],[0],color='lightgrey',label='$\\mathrm{Rate\\ estimate}$'),
]
plt.legend(frameon=False, handles=labels, loc='upper center', fontsize=12)

plt.xlabel('$c_{6}$', fontsize=15)
plt.ylabel('$c_{H}$', fontsize=15)

plt.xlim(-20,20)
plt.ylim(-0.01,0.02)

plt.tick_params(axis='both', labelsize=12)

plt.text(0.04 ,0.12, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=fig.axes[0].transAxes, ha='left', va='bottom', fontsize=12)
plt.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=fig.axes[0].transAxes, ha='left', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig('nll_gg.pdf', bbox_inches='tight')
plt.close()